EDA draft

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
DATA_FOLDER: str = "./data/"
UNICEF_DATA_CSV: str = DATA_FOLDER + "unicef_malawi.csv"
OUTPUT_FOLDER: str = "./output/"

In [2]:
unicef_df = pd.read_csv(UNICEF_DATA_CSV)

# Make target binary
Use map to transform target column "FCF26" to a binary column "FCF26_binary", where 0 = not depressed, and 1 = depressed

In [3]:
unicef_df = pd.read_csv(UNICEF_DATA_CSV)
print(f"FCF26 {unicef_df['FCF26'].isnull().mean() * 100:2f} % nan values")
unicef_df = unicef_df.dropna(subset=["FCF26"])

FCF26 0.782556 % nan values


In [4]:
# depression_map maps values in FCF26 to binary; where 0 = not depressed, and 1 = depressed.
depression_map: dict[str: int] = {
    "NEVER": 0,
    "A FEW TIMES A YEAR": 0,
    "MONTHLY": 0,
    "WEEKLY": 1,
    "DAILY": 1,
}

def map_target_to_binary(
    *,
    df: pd.DataFrame,
    value_to_binary: dict[str, int],
) -> pd.DataFrame:
    df["FCF26_binary"] = df["FCF26"].map(value_to_binary)
    return df

unicef_df = unicef_df[unicef_df["FCF26"] != "NO RESPONSE"]
unicef_df = map_target_to_binary(df=unicef_df, value_to_binary=depression_map)
print(f"FCF26_binary {unicef_df['FCF26_binary'].isnull().mean() * 100:2f} % nan values")
print(unicef_df['FCF26'].unique())

FCF26_binary 0.000000 % nan values
<StringArray>
['NEVER', 'DAILY', 'A FEW TIMES A YEAR', 'WEEKLY', 'MONTHLY']
Length: 5, dtype: str


# Rank correlations to target
We use Cramér's V to measure the association between each feature and the target, removing features with Cramér's V < 0.05, corresponding to less than 0.25% of variance explained individually. While individual features explain modest variance; reflecting the multifactorial and self-reported nature of the data, the combined predictive power of retained features is expected to be substantially higher, as the model learns from many weak signals jointly.

In [5]:
from scipy.stats import chi2_contingency
import numpy as np

def perform_cramers_v(
    *,
    df: pd.DataFrame,
    min_score: float = 0.05,
    target_column_name: str,
    ignore_colums: list,
) -> pd.DataFrame:
    """Performs cramers v test on the columns in the provided 'df' 
    against the 'target_column_name' column and returns a pd.DataFrame
    containing the signficiant features that meet the 'min_score'
    threshold. Ignores the 'ignore_colums'.
    """
    results = []
    df_without_target = df.drop(columns=ignore_colums)
    for col in df_without_target.columns:
        contingency_table = pd.crosstab(df[col], df[target_column_name])
        chi2, p, _, _ = chi2_contingency(contingency_table)
        n = contingency_table.sum().sum()
        min_dim = min(contingency_table.shape) - 1
        cramers_v = np.sqrt(chi2 / (n * min_dim))
        results.append({"feature_name": col, "p_value": p, "cramers_v": cramers_v})
    results_df = pd.DataFrame(results).sort_values("cramers_v", ascending=False)
    return results_df[results_df["cramers_v"] > min_score]

significant_features_df = perform_cramers_v(
    df=unicef_df,
    min_score=0.05,
    target_column_name="FCF26_binary",
    ignore_colums=["FCF26_binary", "FCF26"]

)
print(f"{len(significant_features_df)} features retained")
significant_features_df

29 features retained


,feature_name,p_value,cramers_v
28,wscore,5.303854e-01,0.998658
0,HH1,7.369146e-32,0.366855
10,CL3,9.147104e-03,0.129048
58,LS2,5.400523e-30,0.114343
57,LS1,4.425543e-28,0.104064
27,ethnicity,1.252945e-25,0.102337
80,WS4,2.086852e-02,0.100284
59,LS3,4.184646e-23,0.091711
25,HH7,9.319933e-23,0.088219
44,VT22B,1.747649e-19,0.084207


We remove w-score from our feature set as it represents a measure of child happiness; arguably an alternative target variable rather than a predictor, and its inclusion would risk circular reasoning. We additionally remove ethnicity to prevent the introduction of racial bias into a model that will inform resource allocation decisions. While ethnicity shows a weak but non-negligible association with depression (Cramér's V = 0.1, accounting for ~1% of variance), this association likely reflects proxy effects of socioeconomic status, cultural differences in self-reporting, or experiences of discrimination; factors better captured by other features in the dataset. We note that any future reintroduction of ethnicity would require significant ethical justification. This leaves us with 27 features.

In [6]:
def remove_features_from_df(
    *,
    df: pd.DataFrame,
    features_to_remove: list[str],
) -> pd.DataFrame:
    """Removes the features in 'features_to_remove' from the provided 'df' and returns the df.
    """
    return df[~df["feature_name"].isin(features_to_remove)] # removes features not in features_to_remove.

significant_features_df = remove_features_from_df(
    df=significant_features_df,
    features_to_remove = [
        "ethnicity", # Remove to prevent racial bias.
        "wscore", # Remove because wscore is a direct measure of happiness. Could be considered alternate target.
    ]
)
significant_features: list[str] = significant_features_df["feature_name"].to_list()
significant_features

['HH1',
 'CL3',
 'LS2',
 'LS1',
 'WS4',
 'LS3',
 'HH7',
 'VT22B',
 'MA2',
 'HC4',
 'VT22A',
 'CL13',
 'LS4',
 'WS11',
 'VT22C',
 'HH2',
 'WS7',
 'HC12',
 'MSTATUS',
 'FCD2H',
 'WS1',
 'FCD2J',
 'VT20',
 'HC5',
 'HC8',
 'VT22D',
 'WB4']

We copy our dataframe, keeping only the significant features identified, and the target "FCF26_binary"

In [7]:
unicef_df_filtered = unicef_df[significant_features + ["FCF26_binary"]].copy()

We further look at the percentage of nan values in a column. "CL3" has 68% values nan, "MA2" 23%, "CL13" 15%, "FCD2H" and "FCD2J" 14%, "WS4" 13%.

In [8]:
unicef_df_filtered.isnull().mean().sort_values(ascending=False) * 100

CL3             68.157410
MA2             22.468549
CL13            15.058300
FCD2H           13.769561
FCD2J           13.769561
WS4             13.355324
VT22A            2.140227
LS1              2.140227
VT22B            2.140227
VT22C            2.140227
LS2              2.140227
LS3              2.140227
VT22D            2.140227
WB4              2.140227
VT20             2.140227
LS4              2.140227
HC4              0.000000
HH7              0.000000
HH1              0.000000
WS11             0.000000
MSTATUS          0.000000
HC12             0.000000
WS7              0.000000
HH2              0.000000
HC5              0.000000
WS1              0.000000
HC8              0.000000
FCF26_binary     0.000000
dtype: float64

In [9]:
def identify_columns_of_type_object(df: pd.DataFrame):
    """Identifies the categorical columns that are object data types.
    These columns will need converting into numerical categories such that a model can understand them.
    """
    return df.select_dtypes(include=["object"]).columns.tolist()

categorical_cols = identify_columns_of_type_object(unicef_df_filtered)
print(f"{len(categorical_cols)} categorical object columns found: {categorical_cols}")

23 categorical object columns found: ['LS2', 'LS1', 'WS4', 'LS3', 'HH7', 'VT22B', 'MA2', 'HC4', 'VT22A', 'CL13', 'LS4', 'WS11', 'VT22C', 'WS7', 'HC12', 'MSTATUS', 'FCD2H', 'WS1', 'FCD2J', 'VT20', 'HC5', 'HC8', 'VT22D']


C:\Users\Ed\AppData\Local\Temp\ipykernel_24952\965583777.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  return df.select_dtypes(include=["object"]).columns.tolist()


We now need to transform categories of features into numerical representations, either one-hot encodings, binary, or ordinal.

In [10]:
for col in categorical_cols:
    print(f"\n{col}: {unicef_df_filtered[col].unique()}")


LS2: <StringArray>
['10', '1', '5', '6', '4', '8', nan, '0', '7', '3', '9', 'NO RESPONSE', '2']
Length: 13, dtype: str

LS1: <StringArray>
[         'SOMEWHAT UNHAPPY',            'SOMEWHAT HAPPY',
                'VERY HAPPY', 'NEITHER HAPPY NOR UNHAPPY',
              'VERY UNHAPPY',                         nan,
               'NO RESPONSE']
Length: 7, dtype: str

WS4: <StringArray>
[                   '5.0',                   '30.0',                    '6.0',
                    '8.0',                   '40.0',                      nan,
                   '10.0',                    '7.0',                   '60.0',
                   '15.0',                   '28.0',                   '45.0',
                   '20.0',                   '50.0',                   '70.0',
                   '90.0',                    '2.0',                   '25.0',
                  '120.0',                     'DK',                   '35.0',
                   '26.0',                   '16.0',      

# test / train split

In [11]:
from sklearn.model_selection import train_test_split

x = unicef_df_filtered.drop(columns=["FCF26_binary"])
y = unicef_df_filtered["FCF26_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2, 
    random_state=42, 
    stratify=y,
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

ModuleNotFoundError: No module named 'sklearn'

# Full Pipeline

In [15]:
class FeatureNames:
    """Stores feature names used throughout.
    Does not containt all 87 features in the dataset.
    """
    FCF26 = "FCF26"
    FCF26_BINARY = "FCF26_binary"
    ETHNICITY = "ethnicity"
    WSCORE = "wscore"
    HH7 = "HH7"
    HC4 = "HC4"
    HC5 = "HC5"
    WS11 = "WS11"
    WS1 = "WS1"

feature_config = {
    # LS1: To Mother: mother's happiness level, 4 categories
    "LS1": {
        "map": {
            "VERY UNHAPPY": 0,
            "SOMEWHAT UNHAPPY": 1,
            "NEITHER HAPPY NOR UNHAPPY": 2,
            "SOMEWHAT HAPPY": 3,
            "VERY HAPPY": 4,
            "NO RESPONSE": None,
        },
    },
    # LS2 To Mother: mother's happiness level 0-10
    "LS2": {
        "map": {
            "NO RESPONSE": None,
        },
        "numeric": True,
    },
    # LS3: To Mother: Compared to this time last year, would you say that your life has improved, stayed more or less the same, or worsened, overall?
    "LS3": {
        "map": {
            "WORSENED": 0,
            "MORE OR LESS THE SAME": 1,
            "IMPROVED": 2,
            "NO RESPONSE": None,
        },
    },
    # LS4: To Mother: And in one year from now, do you expect that your life will be better, will be more or less the same, or will be worse, overall?
    "LS4": {
        "map": {
            "WORSE": 0,
            "MORE OR LESS THE SAME": 1,
            "BETTER": 2,
            "NO RESPONSE": None,
        },
    },
    # WS4: How long does it take for members of your household to go there, get water, and come back?
    "WS4": {
        "map": {"DK": None, "MEMBERS DO NOT COLLECT": None, "NO RESPONSE": None},
    },
    # WS7: In the last month, has there been any time when your household did not have sufficient quantities of drinking water?
    "WS7": {
        "map": {
            "NO, ALWAYS SUFFICIENT": 0,
            "YES, AT LEAST ONCE": 1,
            "DK": None,
            "NO RESPONSE": None,
        },
    },
    # VT20: To Mother: How safe do you feel walking alone in your neighbourhood after dark?
    "VT20": {
        "map": {
            "VERY UNSAFE": 0,
            "UNSAFE": 1,
            "NEVER WALK ALONE AFTER DARK": 2,
            "SAFE": 3,
            "VERY SAFE": 4,
            "NO RESPONSE": None,
        },
    },
    # VT22: In the past 12 months, have you personally felt discriminated against or harassed on the basis of the following grounds?
    # VT22A: A) Ethnic or immigration origin?
    "VT22A": {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22B: B) Sex?
    "VT22B": {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22C: C) Sexual orientation?
    "VT22C": {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22D: D) Age?
    "VT22D": {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # MA2: How old is your (husband/partner)?
    "MA2": {"map": {"DK": None, "NO RESPONSE": None}, "numeric": True},
    # CL13: Since last (day of the week), about how many hours did (name) engage in (this activity/these activities), in total?
    "CL13": {"map": {"NO RESPONSE": None}, "numeric": True},
    # HC12: Does any member of your household have a mobile telephone?
    "HC12": {"map": {"NO": 0, "YES": 1, "NO RESPONSE": None}},
    # MSTATUS: marital status of mother
    "MSTATUS": {
        "map": {
            "Never married/in union": 0,
            "Formerly married/in union": 1,
            "Currently married/in union": 2,
        },
    },
    #FCD: Question to child: Adults use certain ways to teach children the 
    # right behaviour or to address a behaviour problem. I will read various
    # methods that are used. Please tell me if you or any other adult in your
    # household has used this method with (name) in the past month.

    # FCD2H: Called (him/her) dumb, lazy or another name like that.
    "FCD2H": {"map": {"NO": 0, "YES": 1, "NO RESPONSE": None}},
    # FCD2J: Hit or slapped (him/her) on the hand, arm, or leg.
    "FCD2J": {"map": {"NO": 0, "YES": 1, "NO RESPONSE": None}},
    # HC8: Does your household have electricity?
    "HC8": {
        "map": {
            "NO": 0,
            "YES, OFF-GRID (GENERATOR/ISOLATED SYSTEM)": 1,
            "YES, INTERCONNECTED GRID": 2,
            "NO RESPONSE": None,
        },
    },
    
}

def apply_feature_config(*, df: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = df.copy()
    for col, rules in config.items():
        if col not in df.columns:
            continue
        if "map" in rules:
            df[col] = df[col].map(rules["map"]).fillna(df[col])
        if rules.get("numeric"):
            df[col] = pd.to_numeric(df[col], errors="coerce")       
    return df

In [ ]:
"""
 ================================
  Feature Extraction and Loading
 ================================
  Section loads data, identifies the top features, manually removes features on further
  inspection. Then we convert all features to numerical representations, either binary,
  one-hot encodings (where a logical order does not make sense), and ordinal values in
  a logical order of significance.
"""
    
# Read unicef data.
unicef_df = pd.read_csv(UNICEF_DATA_CSV)

# Drop rows corresponding to nan values in 'FCF26'. We do this because we will use 'FCF26' to derrive the target variable.
print(f"Dropping {unicef_df[FeatureNames.FCF26].isnull().mean() * 100:2f} % of rows corresponding to nan values in 'FCF26'.")
unicef_df = unicef_df.dropna(subset=[FeatureNames.FCF26])

# depression_map maps categories in the "FCF26" feature to binary values; where 0 = not depressed, and 1 = depressed.
depression_map: dict[str: int] = {
    "NEVER": 0,
    "A FEW TIMES A YEAR": 0,
    "MONTHLY": 0,
    "WEEKLY": 1,
    "DAILY": 1,
}

unicef_df = unicef_df[unicef_df[FeatureNames.FCF26] != "NO RESPONSE"]
unicef_df[FeatureNames.FCF26_BINARY] = unicef_df[FeatureNames.FCF26].map(depression_map)
print(f"Target: {FeatureNames.FCF26_BINARY} has {unicef_df[FeatureNames.FCF26_BINARY].isnull().mean() * 100:2f} % nan values")

# Remove rows where target has value "NO RESPONSE" (Removes 0.78% of total rows)
unicef_df = unicef_df[unicef_df[FeatureNames.FCF26] != "NO RESPONSE"]
unicef_df = map_target_to_binary(df=unicef_df, value_to_binary=depression_map)

# Use Cramer's V score to select features that have a sufficient correlation with the target "FCF26_binary".
significant_features_df = perform_cramers_v(
    df=unicef_df,
    min_score=0.05, # consider p > 0.05 a significant feature.
    target_column_name=FeatureNames.FCF26_BINARY,
    ignore_colums=[
        FeatureNames.FCF26_BINARY, # FCF26_binary is target so remove.
        FeatureNames.FCF26,  # target is derrived from FCF26 so remove.
    ]
)

# Remove some signficant features on further inspection.
significant_features_df = remove_features_from_df(
    df=significant_features_df,
    features_to_remove = [
        FeatureNames.ETHNICITY, # Remove ethnicity feature to prevent racial bias.
        FeatureNames.WSCORE, # Remove wscore feature because it is a direct measure of happiness. Could be considered alternate target.
    ]
)
significant_features: list[str] = significant_features_df["feature_name"].to_list()

# Filter the unicef dataframe, including only significant features and the target.
unicef_df_filtered = unicef_df[significant_features + ["FCF26_binary"]].copy()

# Apply feature config to transform features into a numerical format that our model can understand.
# Nominal Features we one-hot encode.
nominal_features = [
    # HC4: Main material of the dwelling floor.
    FeatureNames.HC4,
    # HC5: Main material of the roof.
    FeatureNames.HC5,
    # HH7: Region
    FeatureNames.HH7,
    # WS1: What is the main source of drinking water used by members of your household?
    FeatureNames.WS1,
    # WS11: What kind of toilet facility do members of your household usually use?
    FeatureNames.WS11,
]
unicef_df_filtered = pd.get_dummies(
    unicef_df_filtered,
    columns=nominal_features,
    drop_first=True, # We set drop_first=True to reduce redundant information, a matrix of all zeroes implies the first category.
)
# For the categorical features where a natural ordering makes sense, we transform them using a map, mapping the category to a 
# numeric value our model will understand.
# unicef_df_filtered = apply_feature_config(df=unicef_df_filtered, config=feature_config)
unicef_df_filtered.head()
null_counts = unicef_df_filtered.isnull().sum()
print(null_counts[null_counts > 0])

Dropping 0.782556 % of rows corresponding to nan values in 'FCF26'.
Target: FCF26_binary has 0.000000 % nan values
CL3      8885
LS2       279
LS1       279
WS4      1741
LS3       279
VT22B     279
MA2      2929
VT22A     279
CL13     1963
LS4       279
VT22C     279
FCD2H    1795
FCD2J    1795
VT20      279
VT22D     279
WB4       279
dtype: int64
